# 02 — Featurization + baseline Random Forest

Prima pipeline completa end-to-end:
1. Featurization (Morgan FP + descrittori RDKit)
2. Scaffold split
3. Random Forest baseline
4. Metriche e plot diagnostici

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from data import load_esol, scaffold_split
from features import featurize_dataframe
from eval import regression_metrics, parity_plot, residual_plot

In [ ]:
df = load_esol(data_dir='../data/raw')
df = df.dropna(subset=['smiles', 'y']).reset_index(drop=True)
print(f'Molecole: {len(df)}')

In [ ]:
# Featurization: Morgan + descrittori RDKit (~2258 feature totali)
X = featurize_dataframe(df, use_morgan=True, use_descriptors=True)
y = df['y'].values
print(f'X shape: {X.shape}')

In [ ]:
# Scaffold split 80/10/10
tr_idx, va_idx, te_idx = scaffold_split(df, frac_train=0.8, frac_valid=0.1, frac_test=0.1)
print(f'Train: {len(tr_idx)}  |  Valid: {len(va_idx)}  |  Test: {len(te_idx)}')

X_tr, X_va, X_te = X[tr_idx], X[va_idx], X[te_idx]
y_tr, y_va, y_te = y[tr_idx], y[va_idx], y[te_idx]

In [ ]:
# Random Forest — iperparametri ragionevoli, no tuning per ora
rf = RandomForestRegressor(
    n_estimators=500,
    max_features='sqrt',
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_tr, y_tr)

pred_va = rf.predict(X_va)
pred_te = rf.predict(X_te)

print('Validation:', regression_metrics(y_va, pred_va))
print('Test:      ', regression_metrics(y_te, pred_te))

In [ ]:
parity_plot(y_te, pred_te, title='RF — Test set', save_path='../results/figures/rf_parity.png')
residual_plot(y_te, pred_te, save_path='../results/figures/rf_residuals.png')

In [ ]:
# Salva metriche su CSV cumulativo
from pathlib import Path
metrics_path = Path('../results/metrics.csv')
row = {'model': 'RandomForest', 'features': 'morgan+rdkit', **regression_metrics(y_te, pred_te)}
if metrics_path.exists():
    pd.concat([pd.read_csv(metrics_path), pd.DataFrame([row])]).to_csv(metrics_path, index=False)
else:
    pd.DataFrame([row]).to_csv(metrics_path, index=False)
print('Saved.')

## Risultati attesi (riferimento)

Su ESOL con scaffold split, una RF ben configurata si attesta tipicamente intorno a **RMSE ~0.95–1.10** sul test. Sotto 1.0 sei già in linea con la letteratura recente per modelli classici. Le GNN scendono a 0.55–0.70.